# Hierarchical Subtopic Analysis
<!-- structured-notebook -->
## Notebook Summary
Purpose: for large parent topics discovered by BERTopic, decompose them into finer-grained subtopics using a clustering strategy optimized for smaller corpora.

Main steps:
- Load a saved BERTopic model and the topic-to-documents mapping.
- Select a parent topic to decompose (from the candidates flagged in `03_topic_ranking_and_narrowing.ipynb`).
- Reduce with PCA(25), cluster with MiniBatchKMeans, then fit a new BERTopic on the sub-corpus.
- Save per-subtopic model, assignments CSV, and doc mapping pickle.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/bertopic_no_embed/` | Produced by `02_preprocessing_and_topic_modelling/01_preprocessing...` |
| Input | `<DATA_ROOT>/Reddit/output/preprocessed-docs.pkl` | Produced by `02_preprocessing_and_topic_modelling/01_preprocessing...` |
| Output | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/subtopics/` | Per-parent-topic subtopic models + doc maps |


# Why PCA + KMeans Instead of UMAP + HDBSCAN
<!-- structured-notebook -->
## Summary
The main pipeline uses UMAP + HDBSCAN, which excels at discovering clusters in large high-dimensional datasets (60K+ documents). Within a single parent topic, the document count is much smaller (hundreds to low thousands) and HDBSCAN becomes unreliable:

- Density estimation is noisy with few points.
- The algorithm may assign most documents to a single cluster or to noise.
- Results are highly sensitive to `min_cluster_size` at small scales.

PCA + MiniBatchKMeans is the alternative:

- **PCA(n_components=25)** — linear dimensionality reduction, stable and interpretable subspace. More deterministic than UMAP for small N.
- **MiniBatchKMeans(n_clusters=K)** — guarantees exactly K subtopics, no noise class. KMeans is well-suited when partitioning a topic into a fixed number of sub-themes.

Third-level (subsubtopic) decomposition was tried but rarely worth the added complexity. See `archive/03_subsubtopic_modeling.ipynb`.


## 1. Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Walk up from the notebook's directory to the repo root (the first ancestor that contains `src/`) and add it to sys.path
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from src.shared_reddit_telegram.text_cleaning import clean_text, is_english, lemmatize_texts, process_docs, dedupe_strings, preprocess_pipeline
from src.shared_reddit_telegram.config import PROJECT_ROOT, REDDIT_DATA, REDDIT_SMALL_DATA
from src.shared_reddit_telegram.topic_utils import explore_subtopics

print(f"Project root: {PROJECT_ROOT}")

## 2. Configuration

Select the parent topic to decompose and set the number of target subtopics.

| Parameter | Purpose |
|---|---|
| `MODEL_PATH` | Path to the saved BERTopic model (without embedding model) |
| `DOC_MAP_PATH` | Path to the topic-to-documents mapping pickle |
| `TOPIC_ID` | Parent topic ID to decompose into subtopics |
| `N_SUBTOPICS` | Target number of subtopics (capped to available docs) |
| `SAVE_DIR` | Directory for subtopic model and output files |

In [ ]:
MODEL_PATH = "../scripts/topic_modelling_v2/round_11/bertopic_no_embed"
DOC_MAP_PATH = "../scripts/topic_modelling_v2/round_11/topic_doc_map.pkl"
TOPIC_ID = 1  # Change this to analyze different parent topics
N_SUBTOPICS = 10
SAVE_DIR = f"../scripts/topic_modelling_v2/round_11/subtopics/{TOPIC_ID}_subtopic_model"

print(f"Parent topic:  {TOPIC_ID}")
print(f"N subtopics:   {N_SUBTOPICS}")
print(f"Model path:    {MODEL_PATH}")
print(f"Doc map path:  {DOC_MAP_PATH}")
print(f"Save dir:      {SAVE_DIR}")

## 3. Why PCA + KMeans for Subtopics

A brief comparison of the two clustering strategies:

| Property | Main pipeline (UMAP + HDBSCAN) | Subtopic pipeline (PCA + KMeans) |
|---|---|---|
| **Scale** | 60K+ documents | 100-3000 documents |
| **Dimensionality reduction** | UMAP (nonlinear, stochastic) | PCA (linear, deterministic) |
| **Clustering** | HDBSCAN (density-based, auto-K) | MiniBatchKMeans (centroid-based, fixed K) |
| **Noise handling** | Assigns outliers to topic -1 | No noise class; all docs assigned |
| **Determinism** | Approximate (UMAP has randomness) | Fully deterministic (with fixed seed) |
| **When to use** | Large, heterogeneous corpus | Small, already-filtered subset |

**PCA(n_components=25)** was chosen because:
- 25 components typically capture 80-90% of variance in sentence embeddings
- It provides a stable subspace even with few hundred documents
- It removes noise dimensions that would confuse KMeans

**MiniBatchKMeans** was chosen over regular KMeans for:
- Faster fitting on moderate-sized corpora
- Near-identical cluster quality
- The `random_state=42` ensures reproducibility

## 4. Run Subtopic Analysis

The `explore_subtopics` function from `shared.topic_utils` handles the full pipeline:
1. Load the parent BERTopic model and topic-doc mapping
2. Extract documents for the target topic
3. Embed them with `all-mpnet-base-v2`
4. Fit a subtopic BERTopic model (PCA + KMeans + c-TF-IDF)
5. Save the subtopic model, assignments, and topic info

In [ ]:
subtopic_model = explore_subtopics(
    topic_model_path=MODEL_PATH,
    topic_doc_map_path=DOC_MAP_PATH,
    topic_id=TOPIC_ID,
    save_dir=SAVE_DIR,
    embedding_model_name="all-mpnet-base-v2",
    n_subtopics=N_SUBTOPICS,
    top_n_words=10,
    visualize=False,  # Set to True to generate interactive HTML visualizations
)

## 5. Inspect Results

Load the subtopic info CSV and examine the top words per subtopic to understand the decomposition.

In [ ]:
# Load subtopic info
subtopic_info_path = os.path.join(SAVE_DIR, f"subtopic_info_topic{TOPIC_ID}.csv")
subtopic_info = pd.read_csv(subtopic_info_path)

print(f"Subtopic decomposition of parent topic {TOPIC_ID}")
print(f"Number of subtopics: {len(subtopic_info)}")
print("=" * 80)
subtopic_info

In [ ]:
# Display top words per subtopic in a readable format
print(f"\nTop words per subtopic (parent topic {TOPIC_ID}):")
print("-" * 80)
for _, row in subtopic_info.iterrows():
    topic_id = row["Topic"]
    count = row["Count"]
    words = row.get("TopWords", "")
    print(f"  Subtopic {topic_id:>3d} ({count:>5,} docs): {words}")

In [ ]:
# Load subtopic assignments to inspect document distribution
assignments_path = os.path.join(SAVE_DIR, f"subtopic_assignments_topic{TOPIC_ID}.csv")
df_assign = pd.read_csv(assignments_path)

print(f"\nSubtopic assignment distribution:")
print(df_assign["subtopic"].value_counts().sort_index())
print(f"\nMean max probability: {df_assign['max_prob'].mean():.3f}")
print(f"Median max probability: {df_assign['max_prob'].median():.3f}")

In [ ]:
# Sample documents from each subtopic
print(f"\nSample documents per subtopic:")
print("=" * 80)
for st_id in sorted(df_assign["subtopic"].unique()):
    subset = df_assign[df_assign["subtopic"] == st_id]
    print(f"\n--- Subtopic {st_id} ({len(subset)} docs) ---")
    for _, row in subset.head(3).iterrows():
        print(f"  [{row['max_prob']:.3f}] {row['doc'][:120]}")

---
<!-- nav-footer -->

← [02 timestamp attachment](../../../../../SocialMedia/Reddit/notebooks/BERTopic/02_preprocessing_and_topic_modelling/02_timestamp_attachment.ipynb) | [Project Overview](../../../../../PROJECT_OVERVIEW.ipynb) | [02 topic merging](../../../../../SocialMedia/Reddit/notebooks/BERTopic/03_topic_refinement/02_topic_merging.ipynb) →
